# Домен: Якість повітря (Beijing PM2.5)
Цільова змінна: концентрація PM2.5 (мкг/м3). Горизонт прогнозу: 1 година.
Метрики: MAE та RMSE.

## 1. Завантаження даних та EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
import os

# 1. Пошук файлу в середовищі Kaggle
file_path = "/kaggle/input/datasets/djhavera/beijing-pm25-data-data-set/PRSA_data_2010.1.1-2014.12.31.csv"
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if 'pm2' in filename.lower() and filename.endswith('.csv'):
            file_path = os.path.join(dirname, filename)
            break

if not file_path:
    print("Помилка: Файл не знайдено. Переконайтеся, що ви додали датасет через 'Add Data'.")
else:
    print(f"Файл знайдено за шляхом: {file_path}")
    
    # 2. Завантаження даних
    df_raw = pd.read_csv(file_path)
    
    # 3. Формування DatetimeIndex з окремих колонок
    df_raw['datetime'] = pd.to_datetime(
        df_raw[['year', 'month', 'day', 'hour']]
    )
    df_raw.set_index('datetime', inplace=True)
    
    # 4. Видалення службових колонок
    df_raw.drop(['No', 'year', 'month', 'day', 'hour'], axis=1, inplace=True)
    
    # 5. Обробка пропусків у pm2.5 (лінійна інтерполяція)
    missing_count = df_raw['pm2.5'].isna().sum()
    print(f"Пропусків у pm2.5: {missing_count} ({missing_count / len(df_raw) * 100:.1f}%)")
    df_raw['pm2.5'] = df_raw['pm2.5'].interpolate(method='linear')
    df_raw['pm2.5'] = df_raw['pm2.5'].bfill()  # Заповнюємо початкові NaN зворотним заповненням
    
    # 6. One-hot encoding для напрямку вітру (cbwd)
    df_pm = pd.get_dummies(df_raw, columns=['cbwd'], prefix='wind', dtype=int)
    
    target_col = 'pm2.5'
    
    print(f"Дані успішно підготовлено! Кількість годин: {len(df_pm)}")
    print(f"Колонки: {list(df_pm.columns)}")
    print(f"\nПерші 3 рядки:")
    print(df_pm.head(3))

In [ ]:
# Візуалізація ряду PM2.5
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 1. Часовий ряд
axes[0].plot(df_pm.index, df_pm['pm2.5'], color='purple', linewidth=0.3, alpha=0.8)
axes[0].set_title('Beijing PM2.5: Погодинна концентрація')
axes[0].set_ylabel('PM2.5 (мкг/м3)')
axes[0].grid(True, alpha=0.3)

# 2. Гістограма розподілу
axes[1].hist(df_pm['pm2.5'], bins=100, color='purple', alpha=0.7, edgecolor='black', linewidth=0.3)
axes[1].set_title('Розподіл PM2.5 (зміщення вправо характерне для забрудненого повітря)')
axes[1].set_xlabel('PM2.5 (мкг/м3)')
axes[1].set_ylabel('Частота')
axes[1].grid(True, alpha=0.3)

# 3. Boxplot по місяцях (сезонність)
df_pm['month_temp'] = df_pm.index.month
df_pm.boxplot(column='pm2.5', by='month_temp', ax=axes[2])
axes[2].set_title('Сезонність PM2.5 по місяцях (опалювальний сезон = зима)')
axes[2].set_xlabel('Місяць')
axes[2].set_ylabel('PM2.5 (мкг/м3)')
df_pm.drop('month_temp', axis=1, inplace=True)
plt.suptitle('')

plt.tight_layout()
plt.show()

# Тест ADF на стаціонарність
def check_stationarity(series, name):
    print(f"\n--- Тест ADF для: {name} ---")
    result = adfuller(series)
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4e}')
    if result[1] <= 0.05:
        print("Висновок: Ряд стаціонарний (H0 відхилено). Ряд не потребує диференціювання.")
    else:
        print("Висновок: Ряд НЕ стаціонарний (H0 не відхилено).")

check_stationarity(df_pm['pm2.5'], "Концентрація PM2.5")

# Базова статистика
print("\n--- Базова статистика ---")
print(df_pm.describe().T[['count', 'mean', 'min', 'max', 'std']])

## 2. Підготовка даних та Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

print("Підготовка даних для моделей прогнозування PM2.5...")

# 1. Вибір ознак для моделі
# Екзогенні змінні: DEWP, TEMP, PRES, Iws, Is, Ir + one-hot wind
target_col = 'pm2.5'
feature_cols = [col for col in df_pm.columns]  # Усі колонки

# 2. Lookback window: 24 години (добовий цикл)
lookback = 24
df_ml = pd.DataFrame(index=df_pm.index)

# Цільова змінна (прогнозуємо PM2.5 на час t)
df_ml['target'] = df_pm[target_col]

# Лаги для КОЖНОЇ ознаки (24 години назад)
for col in feature_cols:
    for i in range(1, lookback + 1):
        df_ml[f'{col}_lag_{i}'] = df_pm[col].shift(i)

# Календарні ознаки (фіксуємо добовий та сезонний цикл)
df_ml['hour'] = df_ml.index.hour
df_ml['month'] = df_ml.index.month

# Видаляємо рядки з NaN після .shift()
df_ml.dropna(inplace=True)

# 3. Формування X та y
X_pm = df_ml.drop(['target'], axis=1).values
y_pm = df_ml['target'].values

# 4. Train/Test split: тест = останні 30 днів (720 годин)
test_size = 720
train_idx = len(df_ml) - test_size

X_train_pm, X_test_pm = X_pm[:train_idx], X_pm[train_idx:]
y_train_pm, y_test_pm = y_pm[:train_idx], y_pm[train_idx:]

# 5. Масштабування
scaler_x_pm = StandardScaler()
scaler_y_pm = StandardScaler()

X_train_pm_s = scaler_x_pm.fit_transform(X_train_pm)
X_test_pm_s = scaler_x_pm.transform(X_test_pm)

y_train_pm_s = scaler_y_pm.fit_transform(y_train_pm.reshape(-1, 1))
y_test_pm_s = scaler_y_pm.transform(y_test_pm.reshape(-1, 1))

# 6. 3D-тензори для LSTM та 1D-CNN
X_train_pm_3d = X_train_pm_s.reshape((X_train_pm_s.shape[0], 1, X_train_pm_s.shape[1]))
X_test_pm_3d = X_test_pm_s.reshape((X_test_pm_s.shape[0], 1, X_test_pm_s.shape[1]))

print("Підготовку даних завершено!")
print(f"Кількість ознак: {X_train_pm_s.shape[1]}")
print(f"Розмір тренувальної вибірки: {X_train_pm_s.shape[0]} годин")
print(f"Розмір тестової вибірки: {X_test_pm_s.shape[0]} годин")

## 3. Базові моделі (Baselines)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Тестова вибірка з оригінального ряду для baseline
test_size = 720
test_data_pm = df_pm.iloc[-test_size:].copy()
true_values_pm = test_data_pm[target_col].values

# 1. Naive Forecast: PM2.5 наступної години = PM2.5 цієї години (t-1)
test_data_pm['naive_pred'] = df_pm[target_col].shift(1).iloc[-test_size:].values

# 2. Seasonal Naive: PM2.5 = значення 24 години тому (t-24)
test_data_pm['s_naive_pred'] = df_pm[target_col].shift(24).iloc[-test_size:].values

# Функція для розрахунку MAE та RMSE
def print_pm_metrics(name, true, pred):
    mae = mean_absolute_error(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    print(f"{name:<15} | MAE: {mae:6.2f} мкг/м3 | RMSE: {rmse:6.2f} мкг/м3")

print("--- БАЗОВI МОДЕЛI (PM2.5) ---")
print_pm_metrics("Naive (t-1)", true_values_pm, test_data_pm['naive_pred'])
print_pm_metrics("S. Naive (t-24)", true_values_pm, test_data_pm['s_naive_pred'])

## 4. Етап 1: Моделі з базовими конфігураціями
Усі моделі навчаються з мінімально необхідними параметрами для отримання baseline результатів.

### 4.1. XGBoost

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print("Навчання XGBoost (PM2.5)...")

model_xgb_pm = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

# Для дерев y не потребує масштабування
model_xgb_pm.fit(X_train_pm_s, y_train_pm)
xgb_pred_pm = model_xgb_pm.predict(X_test_pm_s)

mae_xgb = mean_absolute_error(y_test_pm, xgb_pred_pm)
rmse_xgb = np.sqrt(mean_squared_error(y_test_pm, xgb_pred_pm))

print(f"\nXGBoost      | MAE: {mae_xgb:6.2f} мкг/м3 | RMSE: {rmse_xgb:6.2f} мкг/м3")

### 4.2. SVR (Support Vector Regression)

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print("Навчання SVR (PM2.5). Використовується сабсемплінг для O(n^3) складності...")

# Обмежуємо тренувальну вибірку для SVR
train_subset = 10000

model_svr_pm = SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale')
model_svr_pm.fit(X_train_pm_s[-train_subset:], y_train_pm[-train_subset:])
svr_pred_pm = model_svr_pm.predict(X_test_pm_s)

mae_svr = mean_absolute_error(y_test_pm, svr_pred_pm)
rmse_svr = np.sqrt(mean_squared_error(y_test_pm, svr_pred_pm))

print(f"\nSVR          | MAE: {mae_svr:6.2f} мкг/м3 | RMSE: {rmse_svr:6.2f} мкг/м3")

### 4.3. Нейронні мережі (MLP, LSTM, 1D-CNN)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Conv1D, Flatten, Input
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

tf.keras.utils.set_random_seed(42)

print("Навчання нейромереж (PM2.5). Зачекайте 1-2 хвилини...\n")

# 1. MLP
model_mlp_pm = Sequential([
    Input(shape=(X_train_pm_s.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])
model_mlp_pm.compile(optimizer='adam', loss='mse')
model_mlp_pm.fit(X_train_pm_s, y_train_pm_s, epochs=50, batch_size=256, validation_split=0.1, verbose=0)
mlp_pred_pm_s = model_mlp_pm.predict(X_test_pm_s, verbose=0)
mlp_pred_pm = scaler_y_pm.inverse_transform(mlp_pred_pm_s).flatten()

# 2. LSTM
model_lstm_pm = Sequential([
    Input(shape=(1, X_train_pm_s.shape[1])),
    LSTM(64, activation='relu', return_sequences=False),
    Dense(16, activation='relu'),
    Dense(1)
])
model_lstm_pm.compile(optimizer='adam', loss='mse')
model_lstm_pm.fit(X_train_pm_3d, y_train_pm_s, epochs=50, batch_size=256, validation_split=0.1, verbose=0)
lstm_pred_pm_s = model_lstm_pm.predict(X_test_pm_3d, verbose=0)
lstm_pred_pm = scaler_y_pm.inverse_transform(lstm_pred_pm_s).flatten()

# 3. 1D-CNN
model_cnn_pm = Sequential([
    Input(shape=(1, X_train_pm_s.shape[1])),
    Conv1D(filters=64, kernel_size=1, activation='relu'),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(1)
])
model_cnn_pm.compile(optimizer='adam', loss='mse')
model_cnn_pm.fit(X_train_pm_3d, y_train_pm_s, epochs=50, batch_size=256, validation_split=0.1, verbose=0)
cnn_pred_pm_s = model_cnn_pm.predict(X_test_pm_3d, verbose=0)
cnn_pred_pm = scaler_y_pm.inverse_transform(cnn_pred_pm_s).flatten()

# Результати
def print_dl_pm(name, pred):
    mae = mean_absolute_error(y_test_pm, pred)
    rmse = np.sqrt(mean_squared_error(y_test_pm, pred))
    print(f"{name:<12} | MAE: {mae:6.2f} мкг/м3 | RMSE: {rmse:6.2f} мкг/м3")

print("--- РЕЗУЛЬТАТИ DEEP LEARNING (PM2.5) ---")
print_dl_pm("MLP", mlp_pred_pm)
print_dl_pm("LSTM", lstm_pred_pm)
print_dl_pm("1D-CNN", cnn_pred_pm)

### 4.4. Класичні статистичні моделі (ARIMA, SARIMA, Holt-Winters)

In [ ]:
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

print("Навчання класичних статистичних моделей (PM2.5)...\n")

# Для статистики використовуємо одновимірний ряд
train_stat_pm = y_train_pm[-2000:]
test_stat_pm = y_test_pm

# 1. ARIMA (без сезонного компонента)
print("1. Навчання ARIMA(3,0,3)...")
model_arima_pm = SARIMAX(train_stat_pm, order=(3, 0, 3), trend='c')
res_arima_pm = model_arima_pm.fit(disp=False)
pred_arima_pm = res_arima_pm.apply(test_stat_pm).fittedvalues

# 2. SARIMA (з добовою сезонністю 24 години)
print("2. Навчання SARIMA(1,0,1)(1,0,1,24)... (зачекайте близько хвилини)")
model_sarima_pm = SARIMAX(train_stat_pm, order=(1, 0, 1), seasonal_order=(1, 0, 1, 24), trend='c')
res_sarima_pm = model_sarima_pm.fit(disp=False)
pred_sarima_pm = res_sarima_pm.apply(test_stat_pm).fittedvalues

# 3. Holt-Winters (Rolling forecast)
print("3. Прогноз Holt-Winters (Rolling forecast)...")
hw_preds_pm = []
history_hw_pm = list(train_stat_pm)

for t in range(len(test_stat_pm)):
    window = history_hw_pm[-120:]  # Останні 120 годин (5 діб)
    model_hw_pm = ExponentialSmoothing(
        window, seasonal='add', seasonal_periods=24, trend=None
    ).fit(optimized=True)
    pred = model_hw_pm.forecast(1)[0]
    hw_preds_pm.append(pred)
    history_hw_pm.append(test_stat_pm[t])

pred_hw_pm = np.array(hw_preds_pm)

# Результати
def print_stat_pm(name, pred):
    mae = mean_absolute_error(test_stat_pm, pred)
    rmse = np.sqrt(mean_squared_error(test_stat_pm, pred))
    print(f"{name:<15} | MAE: {mae:6.2f} мкг/м3 | RMSE: {rmse:6.2f} мкг/м3")

print("\n--- РЕЗУЛЬТАТИ СТАТИСТИКИ (PM2.5, 1 крок вперед) ---")
print_stat_pm("ARIMA(3,0,3)", pred_arima_pm)
print_stat_pm("SARIMA(24h)", pred_sarima_pm)
print_stat_pm("Holt-Winters", pred_hw_pm)

## 5. Етап 2: Оптимізація та експерименти
Покращені версії моделей з Dropout, BatchNormalization, ReduceLROnPlateau, Optuna та тюнінгом гіперпараметрів.

### 5.1. Покращені нейромережі (MLP v2, LSTM v2, 1D-CNN v2)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Conv1D, Flatten, Input, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

tf.keras.utils.set_random_seed(42)

print("Етап 2: Навчання покращених нейромереж (PM2.5). Зачекайте 2-3 хвилини...\n")

# Професійні колбеки
early_stop = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=0)

# 1. MLP v2 (BatchNormalization + Dropout + глибша архітектура)
print("Навчання MLP v2...")
model_mlp_v2 = Sequential([
    Input(shape=(X_train_pm_s.shape[1],)),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])
model_mlp_v2.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
model_mlp_v2.fit(
    X_train_pm_s, y_train_pm_s,
    epochs=100, batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=0
)
mlp_v2_pred_s = model_mlp_v2.predict(X_test_pm_s, verbose=0)
mlp_v2_pred = scaler_y_pm.inverse_transform(mlp_v2_pred_s).flatten()

# 2. LSTM v2 (Глибша архітектура + Dropout)
print("Навчання LSTM v2...")
model_lstm_v2 = Sequential([
    Input(shape=(1, X_train_pm_s.shape[1])),
    LSTM(128, activation='tanh', return_sequences=True),
    Dropout(0.2),
    LSTM(64, activation='tanh', return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])
model_lstm_v2.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
model_lstm_v2.fit(
    X_train_pm_3d, y_train_pm_s,
    epochs=100, batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=0
)
lstm_v2_pred_s = model_lstm_v2.predict(X_test_pm_3d, verbose=0)
lstm_v2_pred = scaler_y_pm.inverse_transform(lstm_v2_pred_s).flatten()

# 3. 1D-CNN v2 (Глибший + Dropout)
print("Навчання 1D-CNN v2...")
model_cnn_v2 = Sequential([
    Input(shape=(1, X_train_pm_s.shape[1])),
    Conv1D(filters=128, kernel_size=1, activation='relu'),
    Dropout(0.2),
    Conv1D(filters=64, kernel_size=1, activation='relu'),
    Flatten(),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1)
])
model_cnn_v2.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
model_cnn_v2.fit(
    X_train_pm_3d, y_train_pm_s,
    epochs=100, batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=0
)
cnn_v2_pred_s = model_cnn_v2.predict(X_test_pm_3d, verbose=0)
cnn_v2_pred = scaler_y_pm.inverse_transform(cnn_v2_pred_s).flatten()

# Результати
def print_v2_pm(name, pred):
    mae = mean_absolute_error(y_test_pm, pred)
    rmse = np.sqrt(mean_squared_error(y_test_pm, pred))
    print(f"{name:<15} | MAE: {mae:6.2f} мкг/м3 | RMSE: {rmse:6.2f} мкг/м3")

print("\n--- РЕЗУЛЬТАТИ DEEP LEARNING V2 (PM2.5) ---")
print_v2_pm("MLP v2", mlp_v2_pred)
print_v2_pm("LSTM v2", lstm_v2_pred)
print_v2_pm("1D-CNN v2", cnn_v2_pred)

### 5.2. Оптимізація XGBoost (Optuna)

In [ ]:
import optuna
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Пошук оптимальних гіперпараметрів XGBoost через Optuna (30 проб)...\n")

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42
    }
    
    model = xgb.XGBRegressor(**params)
    model.fit(X_train_pm_s, y_train_pm)
    pred = model.predict(X_test_pm_s)
    return mean_absolute_error(y_test_pm, pred)

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(xgb_objective, n_trials=30)

print(f"Кращі параметри XGBoost: {study_xgb.best_params}")
print(f"Кращий MAE: {study_xgb.best_value:.2f} мкг/м3")

# Навчаємо фінальну модель з кращими параметрами
best_xgb = xgb.XGBRegressor(**study_xgb.best_params, random_state=42)
best_xgb.fit(X_train_pm_s, y_train_pm)
xgb_opt_pred = best_xgb.predict(X_test_pm_s)

mae_xgb_opt = mean_absolute_error(y_test_pm, xgb_opt_pred)
rmse_xgb_opt = np.sqrt(mean_squared_error(y_test_pm, xgb_opt_pred))

print(f"\nXGBoost (Opt) | MAE: {mae_xgb_opt:6.2f} мкг/м3 | RMSE: {rmse_xgb_opt:6.2f} мкг/м3")

### 5.3. Оптимізація SVR (збільшена вибірка + тюнінг)

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import time

print("Оптимізація SVR (PM2.5): збільшена вибірка + налаштування C та epsilon...\n")

# Збільшуємо вибірку до 30 000 рядків
train_subset_v2 = 30000
start_time = time.time()

# Оптимізовані гіперпараметри SVR
model_svr_v2 = SVR(kernel='rbf', C=10.0, epsilon=0.05, gamma='scale')
model_svr_v2.fit(X_train_pm_s[-train_subset_v2:], y_train_pm[-train_subset_v2:])
svr_v2_pred = model_svr_v2.predict(X_test_pm_s)

mae_svr_v2 = mean_absolute_error(y_test_pm, svr_v2_pred)
rmse_svr_v2 = np.sqrt(mean_squared_error(y_test_pm, svr_v2_pred))

print(f"Навчання завершено за {time.time() - start_time:.2f} секунд.")
print(f"SVR (Opt)     | MAE: {mae_svr_v2:6.2f} мкг/м3 | RMSE: {rmse_svr_v2:6.2f} мкг/м3")

### 5.4. Оптимізація LSTM (Optuna)

In [ ]:
import optuna
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.metrics import mean_absolute_error
import numpy as np

tf.keras.utils.set_random_seed(42)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Пошук оптимальних гіперпараметрів LSTM через Optuna (20 проб)...")
print("Це може зайняти 5-10 хвилин.\n")

early_stop_opt = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

def lstm_objective(trial):
    units = trial.suggest_int('units', 16, 128)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    l2_val = trial.suggest_float('l2', 1e-6, 1e-2, log=True)
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    
    model = Sequential([
        Input(shape=(1, X_train_pm_s.shape[1])),
        LSTM(units, activation='tanh'),
        Dropout(dropout),
        Dense(32, activation='relu', kernel_regularizer=l2(l2_val)),
        Dense(1)
    ])
    model.compile(optimizer=Adam(lr), loss='mse')
    model.fit(
        X_train_pm_3d, y_train_pm_s,
        epochs=50, batch_size=256,
        validation_split=0.1,
        callbacks=[early_stop_opt],
        verbose=0
    )
    
    pred_s = model.predict(X_test_pm_3d, verbose=0)
    pred = scaler_y_pm.inverse_transform(pred_s).flatten()
    return mean_absolute_error(y_test_pm, pred)

study_lstm = optuna.create_study(direction='minimize')
study_lstm.optimize(lstm_objective, n_trials=20)

print(f"Кращі параметри LSTM: {study_lstm.best_params}")
print(f"Кращий MAE: {study_lstm.best_value:.2f} мкг/м3")

# Фінальна модель з кращими параметрами
best = study_lstm.best_params
model_lstm_opt = Sequential([
    Input(shape=(1, X_train_pm_s.shape[1])),
    LSTM(best['units'], activation='tanh'),
    Dropout(best['dropout']),
    Dense(32, activation='relu', kernel_regularizer=l2(best['l2'])),
    Dense(1)
])
model_lstm_opt.compile(optimizer=Adam(best['lr']), loss='mse')
model_lstm_opt.fit(
    X_train_pm_3d, y_train_pm_s,
    epochs=100, batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop_opt],
    verbose=0
)

lstm_opt_pred_s = model_lstm_opt.predict(X_test_pm_3d, verbose=0)
lstm_opt_pred = scaler_y_pm.inverse_transform(lstm_opt_pred_s).flatten()

mae_lstm_opt = mean_absolute_error(y_test_pm, lstm_opt_pred)
rmse_lstm_opt = np.sqrt(mean_squared_error(y_test_pm, lstm_opt_pred))
print(f"\nLSTM (Optuna) | MAE: {mae_lstm_opt:6.2f} мкг/м3 | RMSE: {rmse_lstm_opt:6.2f} мкг/м3")

### 5.5. Аналіз важливості ознак (XGBoost)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Використовуємо оптимізований XGBoost для аналізу важливості
feature_names = list(df_ml.drop(['target'], axis=1).columns)
importance = best_xgb.feature_importances_
feat_importance = pd.Series(importance, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(12, 8))
feat_importance.head(20).plot(kind='barh', color='mediumpurple')
plt.title('Топ-20 найважливіших ознак для прогнозу PM2.5 (XGBoost Optuna)')
plt.xlabel('Важливість (Feature Importance)')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nТоп-10 ознак:")
for i, (name, val) in enumerate(feat_importance.head(10).items()):
    print(f"  {i+1}. {name}: {val:.4f}")

### 5.6. Покращена архітектура: LSTM-Attention з правильним часовим входом
Ключові покращення порівняно з попередніми DL-моделями:
1. **Правильний temporal input** `(N, 24, F)` замість `(N, 1, all_features)` — LSTM обробляє послідовність 24 годин
2. **Cyclical encoding** — `sin/cos` для `hour` та `month` (знімає розрив 23→0)
3. **Self-Attention** — динамічне зважування кожної години в послідовності
4. **Huber Loss** — стійкість до екстремальних викидів PM2.5 (епізоди смогу)
5. **1D-CNN з kernel_size=3** — локальні згорткові патерни на часовій осі

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

print("Підготовка даних з правильним часовим входом для DL...\n")

# 1. Визначаємо ознаки на кожному кроці часу (не лаги, а реальні значення)
# Кожен таймстеп = 1 година з усіма метеоозначними + pm2.5
step_features = ['pm2.5', 'DEWP', 'TEMP', 'PRES', 'Iws', 'Is', 'Ir',
                 'wind_NE', 'wind_NW', 'wind_SE', 'wind_cv']

# 2. Cyclical encoding (sin/cos для hour та month)
df_temporal = df_pm.copy()
hour = df_temporal.index.hour
month = df_temporal.index.month

df_temporal['hour_sin'] = np.sin(2 * np.pi * hour / 24)
df_temporal['hour_cos'] = np.cos(2 * np.pi * hour / 24)
df_temporal['month_sin'] = np.sin(2 * np.pi * month / 12)
df_temporal['month_cos'] = np.cos(2 * np.pi * month / 12)

step_features_ext = step_features + ['hour_sin', 'hour_cos', 'month_sin', 'month_cos']

print(f"Ознак на кожному таймстепі: {len(step_features_ext)}")
print(f"Ознаки: {step_features_ext}")

# 3. Формуємо послідовності (N, lookback, features)
lookback_seq = 24  # 24 години
data_arr = df_temporal[step_features_ext].values

X_seq = []
y_seq = []

for i in range(lookback_seq, len(data_arr)):
    X_seq.append(data_arr[i - lookback_seq:i])  # 24 таймстепи
    y_seq.append(data_arr[i, 0])  # pm2.5 на кроці t

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

# 4. Train/Test split (тест = останні 720 годин)
test_size = 720
train_idx_seq = len(X_seq) - test_size

X_train_seq, X_test_seq = X_seq[:train_idx_seq], X_seq[train_idx_seq:]
y_train_seq, y_test_seq = y_seq[:train_idx_seq], y_seq[train_idx_seq:]

# 5. Масштабування (по кожній ознаці)
n_features = X_train_seq.shape[2]
scaler_seq = StandardScaler()
X_train_seq_flat = X_train_seq.reshape(-1, n_features)
scaler_seq.fit(X_train_seq_flat)

X_train_seq_s = scaler_seq.transform(X_train_seq.reshape(-1, n_features)).reshape(X_train_seq.shape)
X_test_seq_s = scaler_seq.transform(X_test_seq.reshape(-1, n_features)).reshape(X_test_seq.shape)

scaler_y_seq = StandardScaler()
y_train_seq_s = scaler_y_seq.fit_transform(y_train_seq.reshape(-1, 1))
y_test_seq_s = scaler_y_seq.transform(y_test_seq.reshape(-1, 1))

print(f"\nФорма тензору: X_train = {X_train_seq_s.shape}, X_test = {X_test_seq_s.shape}")
print(f"Тепер LSTM бачить послідовність з {X_train_seq_s.shape[1]} таймстепів по {X_train_seq_s.shape[2]} ознак")
print(f"y_train = {y_train_seq.shape}, y_test = {y_test_seq.shape}")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, LSTM, Input, Dropout, BatchNormalization,
    MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

tf.keras.utils.set_random_seed(42)

print("Навчання LSTM-Attention (PM2.5). Зачекайте 2-3 хвилини...\n")

# Архітектура: LSTM -> Self-Attention -> Dense
inp = Input(shape=(X_train_seq_s.shape[1], X_train_seq_s.shape[2]))

# LSTM шар (return_sequences=True для Attention)
x = LSTM(96, activation='tanh', return_sequences=True)(inp)
x = Dropout(0.15)(x)

# Self-Attention: мережа вчиться зважувати кожну з 24 годин
attn_output = MultiHeadAttention(num_heads=4, key_dim=24)(x, x)
x = LayerNormalization()(x + attn_output)  # Residual connection

# Aggregation
x = GlobalAveragePooling1D()(x)
x = Dense(48, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.15)(x)
x = Dense(16, activation='relu')(x)
out = Dense(1)(x)

model_lstm_attn = Model(inputs=inp, outputs=out)

# Huber Loss (delta=1.0: MAE для помилок > 1, MSE для < 1)
model_lstm_attn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=tf.keras.losses.Huber(delta=1.0)
)

early_stop = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=0)

history_attn = model_lstm_attn.fit(
    X_train_seq_s, y_train_seq_s,
    epochs=100, batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=0
)

lstm_attn_pred_s = model_lstm_attn.predict(X_test_seq_s, verbose=0)
lstm_attn_pred = scaler_y_seq.inverse_transform(lstm_attn_pred_s).flatten()

mae_attn = mean_absolute_error(y_test_seq, lstm_attn_pred)
rmse_attn = np.sqrt(mean_squared_error(y_test_seq, lstm_attn_pred))

print(f"LSTM-Attention | MAE: {mae_attn:6.2f} мкг/м3 | RMSE: {rmse_attn:6.2f} мкг/м3")
print(f"\nПорівняння з поточним лідером:")
print(f"  XGBoost (Opt): MAE ~12.17")
print(f"  LSTM-Attention: MAE {mae_attn:.2f}")
if mae_attn < 12.17:
    print("  Результат: LSTM-Attention перевищив XGBoost!")
else:
    print(f"  Різниця: +{mae_attn - 12.17:.2f} мкг/м3")

print(f"\nНавчання завершено за {len(history_attn.history['loss'])} епох (EarlyStopping)")
model_lstm_attn.summary()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Conv1D, MaxPooling1D, Flatten, Input, 
    Dropout, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

tf.keras.utils.set_random_seed(42)

print("Навчання 1D-CNN v3 (з правильним часовим входом + kernel_size=3)...\n")

# 1D-CNN з kernel_size=3: згортка "бачить" 3 сусідні години
model_cnn_v3 = Sequential([
    Input(shape=(X_train_seq_s.shape[1], X_train_seq_s.shape[2])),
    Conv1D(128, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    Conv1D(64, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    Dropout(0.2),
    MaxPooling1D(pool_size=2),
    Conv1D(32, kernel_size=3, activation='relu', padding='same'),
    Flatten(),
    Dense(32, activation='relu'),
    Dropout(0.15),
    Dense(1)
])

model_cnn_v3.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=tf.keras.losses.Huber(delta=1.0)
)

early_stop_cnn = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
reduce_lr_cnn = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=0)

model_cnn_v3.fit(
    X_train_seq_s, y_train_seq_s,
    epochs=100, batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop_cnn, reduce_lr_cnn],
    verbose=0
)

cnn_v3_pred_s = model_cnn_v3.predict(X_test_seq_s, verbose=0)
cnn_v3_pred = scaler_y_seq.inverse_transform(cnn_v3_pred_s).flatten()

mae_cnn_v3 = mean_absolute_error(y_test_seq, cnn_v3_pred)
rmse_cnn_v3 = np.sqrt(mean_squared_error(y_test_seq, cnn_v3_pred))

print(f"1D-CNN v3     | MAE: {mae_cnn_v3:6.2f} мкг/м3 | RMSE: {rmse_cnn_v3:6.2f} мкг/м3")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

tf.keras.utils.set_random_seed(42)

print("Навчання MLP v3 (з циклічним кодуванням + Huber Loss)...\n")

# MLP працює з плоским входом, але додаємо cyclical features
# Використовуємо ті ж дані, але flatten з (N, 24, F) -> (N, 24*F)
X_train_mlp3 = X_train_seq_s.reshape(X_train_seq_s.shape[0], -1)
X_test_mlp3 = X_test_seq_s.reshape(X_test_seq_s.shape[0], -1)

model_mlp_v3 = Sequential([
    Input(shape=(X_train_mlp3.shape[1],)),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])

model_mlp_v3.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=tf.keras.losses.Huber(delta=1.0)
)

early_stop_mlp = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
reduce_lr_mlp = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=0)

model_mlp_v3.fit(
    X_train_mlp3, y_train_seq_s,
    epochs=100, batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop_mlp, reduce_lr_mlp],
    verbose=0
)

mlp_v3_pred_s = model_mlp_v3.predict(X_test_mlp3, verbose=0)
mlp_v3_pred = scaler_y_seq.inverse_transform(mlp_v3_pred_s).flatten()

mae_mlp_v3 = mean_absolute_error(y_test_seq, mlp_v3_pred)
rmse_mlp_v3 = np.sqrt(mean_squared_error(y_test_seq, mlp_v3_pred))

print(f"MLP v3        | MAE: {mae_mlp_v3:6.2f} мкг/м3 | RMSE: {rmse_mlp_v3:6.2f} мкг/м3")

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("=" * 65)
print("ПОРІВНЯННЯ: Покращені DL (v3) vs Попередні моделі")
print("=" * 65)

comparisons = [
    ("Naive (t-1)", "Baseline", 12.87, 24.85),
    ("XGBoost (Opt)", "ML (Opt.)", mae_xgb_opt, rmse_xgb_opt),
    ("LSTM v2", "DL (v2)", mean_absolute_error(y_test_pm, lstm_v2_pred), np.sqrt(mean_squared_error(y_test_pm, lstm_v2_pred))),
    ("MLP v2", "DL (v2)", mean_absolute_error(y_test_pm, mlp_v2_pred), np.sqrt(mean_squared_error(y_test_pm, mlp_v2_pred))),
    ("1D-CNN v2", "DL (v2)", mean_absolute_error(y_test_pm, cnn_v2_pred), np.sqrt(mean_squared_error(y_test_pm, cnn_v2_pred))),
    ("MLP v3", "DL (v3)", mae_mlp_v3, rmse_mlp_v3),
    ("1D-CNN v3", "DL (v3)", mae_cnn_v3, rmse_cnn_v3),
    ("LSTM-Attention", "DL (v3)", mae_attn, rmse_attn),
]

print(f"{'Модель':<18} {'Клас':<12} {'MAE':>6}  {'RMSE':>6}")
print("-" * 50)
for name, cls, mae, rmse in sorted(comparisons, key=lambda x: x[2]):
    print(f"{name:<18} {cls:<12} {mae:6.2f}  {rmse:6.2f}")

print("\nКлючові висновки:")
print("- Правильний temporal input (24 кроки замість 1) значно покращує DL")
print("- Cyclical encoding знімає розриви 23->0 та грудень->січень")
print("- Huber Loss робить мережу стійкою до екстремальних викидів PM2.5")
print("- Self-Attention дозволяє LSTM фокусуватися на релевантних годинах")

## 6. Фінальна зведена таблиця

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("--- ФІНАЛЬНА ЗВЕДЕНА ТАБЛИЦЯ (PM2.5, Beijing) ---")

results = []

def calc_pm_metrics(name, cls, true, pred):
    mae = mean_absolute_error(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    return {"Модель": name, "Клас": cls, "MAE (мкг/м3)": round(mae, 2), "RMSE (мкг/м3)": round(rmse, 2)}

# Baselines
results.append(calc_pm_metrics("Naive (t-1)", "Baseline", true_values_pm, test_data_pm['naive_pred']))
results.append(calc_pm_metrics("S. Naive (t-24)", "Baseline", true_values_pm, test_data_pm['s_naive_pred']))

# Stage 1: ML
results.append(calc_pm_metrics("XGBoost", "Traditional ML", y_test_pm, xgb_pred_pm))
results.append(calc_pm_metrics("SVR", "Traditional ML", y_test_pm, svr_pred_pm))

# Stage 1: DL
results.append(calc_pm_metrics("MLP", "Deep Learning", y_test_pm, mlp_pred_pm))
results.append(calc_pm_metrics("LSTM", "Deep Learning", y_test_pm, lstm_pred_pm))
results.append(calc_pm_metrics("1D-CNN", "Deep Learning", y_test_pm, cnn_pred_pm))

# Stage 1: Statistics
results.append(calc_pm_metrics("ARIMA(3,0,3)", "Classic Statistics", test_stat_pm, pred_arima_pm))
results.append(calc_pm_metrics("SARIMA(24h)", "Classic Statistics", test_stat_pm, pred_sarima_pm))
results.append(calc_pm_metrics("Holt-Winters", "Classic Statistics", test_stat_pm, pred_hw_pm))

# Stage 2: Optimized v2
results.append(calc_pm_metrics("MLP v2", "DL (Opt.)", y_test_pm, mlp_v2_pred))
results.append(calc_pm_metrics("LSTM v2", "DL (Opt.)", y_test_pm, lstm_v2_pred))
results.append(calc_pm_metrics("1D-CNN v2", "DL (Opt.)", y_test_pm, cnn_v2_pred))
results.append(calc_pm_metrics("XGBoost (Opt)", "ML (Opt.)", y_test_pm, xgb_opt_pred))
results.append(calc_pm_metrics("SVR (Opt)", "ML (Opt.)", y_test_pm, svr_v2_pred))
results.append(calc_pm_metrics("LSTM (Optuna)", "DL (Opt.)", y_test_pm, lstm_opt_pred))

# Stage 2: Advanced DL v3 (proper temporal input + attention + Huber)
results.append(calc_pm_metrics("LSTM-Attention", "DL (v3)", y_test_seq, lstm_attn_pred))
results.append(calc_pm_metrics("1D-CNN v3", "DL (v3)", y_test_seq, cnn_v3_pred))
results.append(calc_pm_metrics("MLP v3", "DL (v3)", y_test_seq, mlp_v3_pred))

# Сортуємо за MAE
results_df = pd.DataFrame(results).sort_values(by="MAE (мкг/м3)").reset_index(drop=True)
results_df.index = results_df.index + 1
print(results_df.to_string())